# routes.sessions

> Session management route handlers — list, create, delete, rename, and resume.

This module provides `init_session_router()` which returns an `APIRouter` and a dict of route handlers. The top-level `init_session_manager_routers()` (in `routes.init`) calls this and plugs the results into the `SessionManagementResult` dataclass.

Routes:

| Route | Method | Purpose |
|:------|:-------|:--------|
| `/management_page` | GET | Full session manager page (header + list) |
| `/list_sessions` | GET | Session list fragment (used for refreshes) |
| `/create_session` | POST | Mint a new session, set it active, return refreshed list |
| `/delete_session` | POST | Delete a session, return refreshed list OOB |
| `/rename_session` | POST | Update a session's label, return refreshed list OOB |
| `/resume_session` | POST | Set the session active and redirect to the workflow URL |

In [ ]:
#| default_exp routes.sessions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Callable, Dict, Tuple

from fasthtml.common import APIRouter
from starlette.responses import Response

from cjm_fasthtml_interactions.core.state_store import set_session_id, get_session_id

from cjm_fasthtml_workflow_session_management.models import SessionManagementUrls
from cjm_fasthtml_workflow_session_management.services.management import SessionManagementService

## Debug Flag

In [ ]:
#| export
DEBUG_SESSION_ROUTES = False # Enable for verbose session route logging

## `init_session_router`

Creates all session-management routes under a single `APIRouter`. Takes the service, URL bundle (populated by the caller after route registration), a `refresh_items_oob` callback for mutations that need to return OOB updates, and a `workflow_url` for the resume redirect target.

The `refresh_items_oob()` callback is the standard pattern from the reference library: after a state mutation, it refreshes the items and returns a tuple of OOB elements (VC rebuild + any extra components) which HTMX swaps into place.

In [ ]:
#| export
def init_session_router(
    service:SessionManagementService, # Service for session CRUD
    prefix:str, # Route prefix (e.g., "/manage/sessions")
    urls:SessionManagementUrls, # URL bundle (populated by caller after init)
    workflow_url:str, # Where to redirect after resume
    refresh_items:Callable, # (request=None) -> reload items from service
    refresh_items_oob:Callable, # (request=None) -> refresh + OOB tuple for HTMX response
    render_page:Callable, # () -> full session manager page
    render_list:Callable, # () -> session list component
) -> Tuple[APIRouter, Dict[str, Callable]]: # (router, routes dict)
    """Initialize session management routes (list, create, delete, rename, resume)."""
    router = APIRouter(prefix=prefix)
    routes:Dict[str, Callable] = {}
    
    @router
    async def management_page(request):
        """Return the full session manager page."""
        if DEBUG_SESSION_ROUTES:
            print("[ROUTES] management_page called")
        refresh_items(request=request)
        return render_page()
    routes["management_page"] = management_page
    
    @router
    async def list_sessions(request):
        """Return just the session list fragment."""
        if DEBUG_SESSION_ROUTES:
            print("[ROUTES] list_sessions called")
        refresh_items(request=request)
        return render_list()
    routes["list_sessions"] = list_sessions
    
    @router.post
    async def create_session(request):
        """Mint a new session, set it active, and return the refreshed list."""
        sess = request.session
        new_id = service.create_session()
        set_session_id(sess, new_id)
        if DEBUG_SESSION_ROUTES:
            print(f"[ROUTES] create_session -> {new_id[:8]}...")
        return refresh_items_oob(request=request)
    routes["create_session"] = create_session
    
    @router.post
    async def delete_session(request):
        """Delete a session and return the refreshed list OOB."""
        form_data = await request.form()
        session_id = form_data.get("session_id", "")
        if DEBUG_SESSION_ROUTES:
            print(f"[ROUTES] delete_session: {session_id[:8] if session_id else '<none>'}...")
        if session_id:
            service.delete_session(session_id)
            # If the active session was deleted, clear the pointer so the next visit mints a fresh one.
            sess = request.session
            if get_session_id(sess) == session_id:
                # Pick the most recently updated remaining session, or mint a new one.
                remaining = service.list_sessions()
                if remaining:
                    set_session_id(sess, remaining[0].summary.session_id)
                else:
                    # No sessions left — mint a new empty one so the workflow has state to resume.
                    fresh_id = service.create_session()
                    set_session_id(sess, fresh_id)
        return refresh_items_oob(request=request)
    routes["delete_session"] = delete_session
    
    @router.post
    async def rename_session(request):
        """Rename a session and return the refreshed list OOB."""
        form_data = await request.form()
        session_id = form_data.get("session_id", "")
        label = form_data.get("label", "") or None  # empty string → clear
        if DEBUG_SESSION_ROUTES:
            print(f"[ROUTES] rename_session: {session_id[:8] if session_id else '<none>'}... -> {label!r}")
        if session_id:
            service.rename_session(session_id, label)
        return refresh_items_oob(request=request)
    routes["rename_session"] = rename_session
    
    @router.post
    async def resume_session(request):
        """Set the session active and redirect the browser to the workflow URL."""
        form_data = await request.form()
        session_id = form_data.get("session_id", "")
        if DEBUG_SESSION_ROUTES:
            print(f"[ROUTES] resume_session: {session_id[:8] if session_id else '<none>'}... -> {workflow_url}")
        if session_id and service.session_exists(session_id):
            set_session_id(request.session, session_id)
        # HX-Redirect tells HTMX to trigger a full-page navigation.
        return Response(status_code=204, headers={"HX-Redirect": workflow_url})
    routes["resume_session"] = resume_session
    
    return router, routes

In [ ]:
# Smoke test: confirm the router assembles without errors.
import tempfile, os
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

_tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_tmp_path = _tmp.name
_tmp.close()
_store = SQLiteWorkflowStateStore(_tmp_path)
_svc = SessionManagementService(_store, "test_flow")

_urls = SessionManagementUrls(
    management_page="", list_sessions="", session_detail="",
    create_session="", delete_session="", rename_session="", resume_session="",
)

_refresh_items = lambda: None
_refresh_items_oob = lambda: ()
_render_page = lambda: None
_render_list = lambda: None

router, routes = init_session_router(
    service=_svc,
    prefix="/m/sessions",
    urls=_urls,
    workflow_url="/workflow/",
    refresh_items=_refresh_items,
    refresh_items_oob=_refresh_items_oob,
    render_page=_render_page,
    render_list=_render_list,
)

expected_routes = {
    "management_page", "list_sessions",
    "create_session", "delete_session", "rename_session", "resume_session",
}
assert set(routes.keys()) == expected_routes
assert router is not None

os.unlink(_tmp_path)
print(f"Session router OK: {sorted(routes.keys())}")

Session router OK: ['create_session', 'delete_session', 'list_sessions', 'management_page', 'rename_session', 'resume_session']


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()